# Cleaned Odyssey's text from Robert Fitzgerald's translation

This is the thirds version. The previous two copies of the text were extremely difficult to clean programmatically. The errata were to varied and copiously disseminated. Case by case, and intance by intance amendment was needed. Because this project focuses on data science not on digital philological work, I decided to find a third copy hoping the prunning can be done using only rule-based methods. This is it (?).

### Before

- Scanned from PDF. Here are some characteristics. 
- Cleanish but
    * Subtitle scheme:  
        `BOOK II`  
        `A HERO'S SON AWAKENS`  
        `[empty line]`
    * Intro and ToC
    * Glossary and notes at the end
    * Separated by books

### After
- Only the text 
- Keep book numbers: 'Book I',…'Book XXIV'
- No numeral digits (if in book title, changed to Roman)

Curiously, Fitzgerald is the **only translator to use "book"** so far. 

In [ ]:
translator = "Lattimore"
filepath = f"/Users/debr/odysseys_en/raw_txts/Odyssey_{translator}.txt"

# Define start and end markers
start_marker = "BOOK I"
end_marker = "daughter of Zeus of the aegis, who had likened herself in appearance and\nvoice to Mentor."

In [ ]:
def extract_text_between_markers(file_path, start_marker, end_marker, skip_empty_lines=True):
    """
    Extract text between start and end markers from a text file.
    
    Args:
        file_path (str): Path to the text file
        start_marker (str): Text that marks the beginning of the section to extract
        end_marker (str): Text that marks the end of the section to extract
        skip_empty_lines (bool): Whether to skip empty lines in the output
    
    Returns:
        list: List of strings, each representing a line in the extracted text
    """
    # Read the entire file content
    with open(file_path, "r", encoding="utf-8") as inputfile:
        file_content = inputfile.read()
    
    # Split into lines for exact matching
    all_lines = file_content.splitlines()
    
    # Find the start and end line indices
    start_index = -1 # Initialize with a value that indicates the marker wasn't found
    end_index = -1 # Initialize with a value that indicates the marker wasn't found
    
    for i, line in enumerate(all_lines):
        if line.strip() == start_marker and start_index == -1: # Only find the first occurrence of the start marker
            start_index = i # Mark the first occurrence of the start marker
        elif line.strip() == end_marker and start_index != -1: # Only find the last occurrence of the end marker after the start marker
            end_index = i # Mark the last occurrence of the end marker
            break 
    
    # Handle cases where markers aren't found
    if start_index == -1: # If the start marker isn't found
        print(f"Warning: Exact start marker '{start_marker}' not found in the file.")
        start_index = 0
    
    if end_index == -1:
        print(f"Warning: Exact end marker '{end_marker}' not found in the file.")
        end_index = len(all_lines) - 1
    
    # Extract the lines between markers (inclusive)
    extracted_lines = all_lines[start_index:end_index+1]
    
    # Filter out empty lines if requested
    if skip_empty_lines:
        extracted_lines = [line for line in extracted_lines if line.strip()]
    
    return extracted_lines

def extract_text_between_markers_many_lines(file_path, start_marker, end_marker, skip_empty_lines=True):
    """
    Extract text between start and end markers from a text file.
    
    Args:
        file_path (str): Path to the text file
        start_marker (str): Text that marks the beginning of the section to extract
        end_marker (str): Text that marks the end of the section to extract
        skip_empty_lines (bool): Whether to skip empty lines in the output
    
    Returns:
        list: List of strings, each representing a line in the extracted text
    """
    # Read the entire file content
    with open(file_path, "r", encoding="utf-8") as inputfile:
        file_content = inputfile.read()
    
    # Find the start and end positions
    start_pos = file_content.find(start_marker)
    end_pos = file_content.find(end_marker)
    
    # Handle cases where markers aren't found
    if start_pos == -1:
        print(f"Warning: Start marker '{start_marker}' not found in the file.")
        start_pos = 0
    else:
        # Include the start marker in the output
        start_pos = start_pos
    
    if end_pos == -1:
        print(f"Warning: End marker '{end_marker}' not found in the file.")
        end_pos = len(file_content)
    else:
        # Include the end marker in the output
        end_pos = end_pos + len(end_marker)
    
    # Extract the text between markers
    extracted_text = file_content[start_pos:end_pos]
    
    # Split the extracted text into lines
    lines = extracted_text.splitlines()
    
    # Filter out empty lines if requested
    if skip_empty_lines:
        lines = [line for line in lines if line.strip()]
    
    return lines

In [ ]:
# Extract the text One line subtitle
extracted_lines = extract_text_between_markers(filepath, start_marker, end_marker)
# Many lines
# extracted_lines = extract_text_between_markers_manylines(filepath, start_marker, end_marker)

# Verify by printing the end of the extracted text
print(f"Tell me, Python, how {translator}'s Odyssey starts:")
print("\n".join(extracted_lines[:4]))

print(f"\nO, but tell me, Python, how {translator}'s Odyssey ends:")
print("\n".join(extracted_lines[-3:]))

In [ ]:
# Step 3: Find subtitle-couplets after "Book…" and remove them
import re
def process_books(lines):
    cleaned_books = []
    current_book = []
    
    for line in lines:
        line = line.strip()  # Remove leading/trailing spaces

        # Skip completely empty lines
        if not line:
            continue  

        # Identify the start of a new book
        if line.startswith("Book "):
            if current_book:  # Process the previous book if it exists
                cleaned_books.extend(process_book_section(current_book))
            current_book = [line]  # Start a new book entry
        else:
            current_book.append(line)
    
    # Process the last book
    if current_book:
        cleaned_books.extend(process_book_section(current_book))

    return cleaned_books

def process_book_section(book_lines):
    """
    Extract the first 4 lines of a book while:
    - Removing all empty lines after "Book …"
    - Pruning lines 2 & 3
    - Keeping all remaining content
    """
    result = []
    result.append(book_lines[0])  # Keep the book title

    # Remove empty lines after "Book …"
    filtered_lines = [line for line in book_lines[1:] if line.strip()]

    # Ensure at least 4 lines exist before pruning
    if len(filtered_lines) > 0:
        result.append("")  # Prune line 2
    if len(filtered_lines) > 1:
        result.append("")  # Prune line 3
    if len(filtered_lines) > 2:
        result.extend(filtered_lines[2:])  # Keep the rest of the book

    return result

skimmed_lines = process_books(extracted_lines)

# Checking and failing to prune empty lines
print("\n".join(skimmed_lines[:5]))

In [ ]:
len(skimmed_lines)

In [ ]:
# Because those persistent empty lines, brute force to remove them
filtered_lines  = [line for line in skimmed_lines if line.strip()]
print(filtered_lines)

In [ ]:
# Extrated lines to text
final_text = "\n".join(filtered_lines)

# Write the cleaned content to a new file for further processing
output_filepath = f"/Users/debr/odysseys_en/cleaned_txts/Odyssey_{translator}_cleaned.txt"
with open(output_filepath, "w", encoding="utf-8") as outputfile:
    outputfile.writelines(final_text)